# 02 - MLP (Multi-Layer Perceptron)

We're moving from bigrams to MLPs.
The bigram model was cool but dumb. It only looked one character back. 
The MLP fixes this. Theres a context window (ex. 3 characters) and it actually has something to work with. 
It learns embeddings for each character, passes them through a hidden layer, and outputs probabilities for what comes next.
This is where things start feeling like real deep learning.

In [5]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
import random
import pandas as pd

In [10]:
df = pd.read_parquet('data/0000.parquet')
comments = df['Comment'].dropna().tolist()
comments = [c.lower() for c in comments]

text = ''.join(comments)
chars = sorted(list(set(text)))
vocab_size = len(chars)

len(comments)

25430

In [7]:
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['|'] = 0
itos = {i:s for s,i in stoi.items()}    

In [11]:
random.seed(42)
random.shuffle(comments)

n1 = int(0.8 * len(comments))
n2 = int(0.9 * len(comments))

train = comments[:n1]
val = comments[n1:n2]
test = comments[n2:]

print(len(train))
print(len(val))
print(len(test))


20344
2543
2543


In [12]:
# Building the dataset
block_size = 3
def build_dataset(comments):
    X, Y = [], []
    for comment in comments: 
        context = [0] * block_size
        for ch in comment: 
            ix =stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
        X.append(context)
        Y.append(0)
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y

In [ ]:
Xtr, Ytr   = build_dataset(train)
Xval, Yval = build_dataset(val)
Xte, Yte   = build_dataset(test)

print(Xtr.shape, Ytr.shape)
print(Xval.shape, Yval.shape)
print(Xte.shape, Yte.shape)


torch.Size([3118732, 3]) torch.Size([3118732])
torch.Size([388886, 3]) torch.Size([388886])
torch.Size([395959, 3]) torch.Size([395959])


In [14]:
Xtr[:10]

tensor([[ 0,  0,  0],
        [ 0,  0, 47],
        [ 0, 47, 35],
        [47, 35, 32],
        [35, 32,  1],
        [32,  1, 28],
        [ 1, 28, 47],
        [28, 47, 47],
        [47, 47, 28],
        [47, 28, 30]])

### Embedding : 

